# 06. Коллаборативные модели с подбором параметров (E1)

Контекст — `решения.md`: 3.2 (модели 4–6), 3.1 (оценка), 3.7 (E1).

- **Данные — как в 03 и 05:** подвыборка 20% пользователей (seed 42) со всей историей, книги с ≥ 5 оценками в подвыборке;
  обучение — до 2021, валидация — 2021 год, тест — с 2022. Метрики @10 из `ml/src/eval/`, разрезы «все / старые / новые».
- **Модели** (`ml/src/cf/models.py`): ItemKNN (cosine / BM25 / TF-IDF), ALS, ALS с признаками книг (раздел, подраздел,
  частые авторы). Коллаборативные модели выбирают из книг обучения; content-модели (2–3) — из всего каталога без книг
  «из одного названия», как в 05.
- **Подбор** (`ml/src/cf/tune.py`): Optuna (TPE, seed 42), цель — NDCG@10 «все» на валидации; ItemKNN — 30 попыток,
  ALS — 30, ALS с признаками — 20. Попытки сохраняются в `models/optuna/cf.db` — прерванный подбор
  продолжается с того же места.
- **Тест — один раз:** лучшие параметры с валидации, обучение на обучении + валидации (до 2022), проверка на 2022–2023.
- **Правило кандидатов (с 04.11, во всех таблицах):** ни одна модель не рекомендует книги с 1–4 оценками в обучении
  (`eval.data.drop_rare`): подвыборка оставляет книги с ≥ 5 оценками за всё время, и такая книга попала в данные только
  благодаря будущим оценкам. Модели выдают топ-30, после правила остаются первые 10. Подбор (раздел 2) шёл без правила —
  параметры не перебирались заново.

In [1]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from implicit.cpu.als import AlternatingLeastSquares
from rectools import Columns
from rectools.models import ImplicitALSWrapperModel, PopularModel

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent / "src"))
from cf.models import SEED, fit_recommend
from cf.tune import best_params, trials_table, tune
from content.embeddings import catalog, load_books
from content.recommend import recommend, tfidf_vectors
from eval.data import TEST_START, VAL_START, drop_rare, first_seen, known_at, load_ratings, subsample_users, time_split, to_dataset
from eval.metrics import K, eval_users, evaluate, paired_bootstrap, with_ci
from eval.search import informative

REPORTS = Path.cwd().parent / "reports" / "metrics"
pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_colwidth = 120
K_GEN = K + 20   # запас под правило кандидатов (drop_rare)

ratings = load_ratings()
first = first_seen(ratings)   # дата первой оценки книги по всем данным — для каталога «известные на дату» (раздел 5)
sub = subsample_users(ratings, 0.2, seed=SEED)
train, val, test = time_split(sub)
users_val = eval_users(train, val)

books = load_books(["parent_asin", "author", "text", "features", "categories"])
ids_all = books["parent_asin"].to_numpy()
allowed_all = informative(books)
X_tfidf, _ = tfidf_vectors(books["text"])
E5 = np.asarray(catalog("e5-base"))
print(f"Пользователей для проверки на валидации: {len(users_val):,}; книг обучения: {train[Columns.Item].nunique():,}")

Пользователей для проверки на валидации: 21,350; книг обучения: 83,639


## 1. Модели без подбора (валидация)

Popularity и ALS по умолчанию — как в 03; content-модели — как в 05 (e5-base — модель, выбранная для поиска, 3.5).

In [2]:
results, per_user, timing = {"val": [], "test": []}, {"val": {}, "test": {}}, []

def popular(tr, users):
    ds = to_dataset(tr, weighted=False)
    m = PopularModel(popularity="n_users").fit(ds)
    return m.recommend(users, ds, k=K_GEN, filter_viewed=True, on_unsupported_targets="ignore")

def als_default(tr, users):
    ds = to_dataset(tr, weighted=False)
    m = ImplicitALSWrapperModel(AlternatingLeastSquares(factors=64, iterations=15, random_state=SEED, num_threads=0)).fit(ds)
    return m.recommend(users, ds, k=K_GEN, filter_viewed=True, on_unsupported_targets="ignore")

BASE = {
    "1 Popularity": popular,
    "2 Content TF-IDF": lambda tr, users: recommend(X_tfidf, ids_all, tr, users, k=K_GEN, allowed=allowed_all),
    "3 Content e5-base": lambda tr, users: recommend(E5, ids_all, tr, users, k=K_GEN, allowed=allowed_all),
    "ALS по умолчанию": als_default,
}

def record(part_name, name, reco, seconds, tr, part, users):
    reco = drop_rare(reco, tr, k=K)   # правило кандидатов
    table, pu = evaluate(reco, tr, part, users)
    results[part_name].append(with_ci(table, pu).assign(модель=name))
    per_user[part_name][name] = pu
    timing.append({"часть": part_name, "модель": name, "обучение + рекомендации, с": round(seconds, 1)})
    print(f"{name}: {seconds:.0f} с, NDCG@10 все / старые / новые: " + " / ".join(f"{v:.4f}" for v in table["NDCG@10"]))

for name, make in BASE.items():
    t0 = time.time()
    reco = make(train, users_val)
    record("val", name, reco, time.time() - t0, train, val, users_val)

1 Popularity: 1 с, NDCG@10 все / старые / новые: 0.0024 / 0.0035 / 0.0000


2 Content TF-IDF: 338 с, NDCG@10 все / старые / новые: 0.0107 / 0.0054 / 0.0176


3 Content e5-base: 37 с, NDCG@10 все / старые / новые: 0.0099 / 0.0049 / 0.0167


ALS по умолчанию: 16 с, NDCG@10 все / старые / новые: 0.0042 / 0.0061 / 0.0000


## 2. Подбор параметров на валидации

In [3]:
TRIALS = {'knn': 30, 'als': 30, 'als_features': 20}
NAMES = {"knn": "4 ItemKNN", "als": "5 ALS", "als_features": "6 ALS + признаки"}
studies, tune_time = {}, {}
for kind, n in TRIALS.items():
    t0 = time.time()
    studies[kind] = tune(kind, train, val, users_val, books, n_trials=n)
    tune_time[kind] = time.time() - t0
    tab = trials_table(studies[kind], NAMES[kind])
    print(f"{NAMES[kind]}: {len(tab)} попыток, сумма {tab['секунд'].sum() / 60:.1f} мин (этот запуск — {tune_time[kind] / 60:.1f} мин), "
          f"лучшая NDCG@10 {tab['NDCG@10'].iloc[0]:.4f}")
    display(tab.head(5))

trials = pd.concat([trials_table(s, NAMES[k]) for k, s in studies.items()], ignore_index=True)
trials.to_csv(REPORTS / "cf_trials.csv", index=False, encoding="utf-8")
BEST = {k: best_params(s) for k, s in studies.items()}
BEST

4 ItemKNN: 30 попыток, сумма 1.8 мин (этот запуск — 0.1 мин), лучшая NDCG@10 0.0111


,модель,попытка,NDCG@10,NDCG@10 старые,Recall@10,секунд,параметры
0,4 ItemKNN,29,0.0111,0.0170,0.0167,4.6000,"{""similarity"": ""bm25"", ""K"": 565, ""weighted"": true, ""K1"": 0.3355838788284944, ""B"": 0.28920875241277766}"
1,4 ItemKNN,12,0.0110,0.0169,0.0159,5.9000,"{""similarity"": ""bm25"", ""K"": 841, ""weighted"": true, ""K1"": 1.2039469758027384, ""B"": 0.20323059853518888}"
2,4 ItemKNN,27,0.0110,0.0168,0.0161,4.5000,"{""similarity"": ""bm25"", ""K"": 529, ""weighted"": true, ""K1"": 0.5405910112042486, ""B"": 0.29619441196177576}"
3,4 ItemKNN,21,0.0109,0.0168,0.0157,3.8000,"{""similarity"": ""bm25"", ""K"": 399, ""weighted"": true, ""K1"": 1.3567251944663423, ""B"": 0.20431665489402417}"
4,4 ItemKNN,22,0.0109,0.0168,0.0158,3.7000,"{""similarity"": ""bm25"", ""K"": 389, ""weighted"": true, ""K1"": 1.202707435397148, ""B"": 0.20688625437905295}"


5 ALS: 30 попыток, сумма 9.9 мин (этот запуск — 0.0 мин), лучшая NDCG@10 0.0083


,модель,попытка,NDCG@10,NDCG@10 старые,Recall@10,секунд,параметры
0,5 ALS,25,0.0083,0.0122,0.0134,22.8000,"{""factors"": 128, ""regularization"": 0.0013995874675233253, ""alpha"": 82.20322733155712, ""iterations"": 23, ""weighted"": ..."
1,5 ALS,28,0.0082,0.0122,0.0134,26.6000,"{""factors"": 128, ""regularization"": 0.025291875234566087, ""alpha"": 64.40206928492441, ""iterations"": 26, ""weighted"": t..."
2,5 ALS,21,0.0082,0.0121,0.0132,17.8000,"{""factors"": 128, ""regularization"": 0.001357041079659927, ""alpha"": 52.24365692618301, ""iterations"": 21, ""weighted"": f..."
3,5 ALS,23,0.0082,0.0121,0.0134,19.5000,"{""factors"": 128, ""regularization"": 0.001090382191512855, ""alpha"": 65.79780157310795, ""iterations"": 17, ""weighted"": f..."
4,5 ALS,24,0.0081,0.0120,0.0132,21.1000,"{""factors"": 128, ""regularization"": 0.004048071028914759, ""alpha"": 42.751757866859414, ""iterations"": 19, ""weighted"": ..."


6 ALS + признаки: 20 попыток, сумма 44.0 мин (этот запуск — 0.0 мин), лучшая NDCG@10 0.0086


,модель,попытка,NDCG@10,NDCG@10 старые,Recall@10,секунд,параметры
0,6 ALS + признаки,11,0.0086,0.0126,0.0133,53.3000,"{""factors"": 32, ""regularization"": 0.0064398121586677305, ""alpha"": 68.3964886529918, ""iterations"": 15, ""weighted"": fa..."
1,6 ALS + признаки,6,0.0086,0.0126,0.0132,53.2000,"{""factors"": 32, ""regularization"": 0.01998634077852888, ""alpha"": 23.86798222334764, ""iterations"": 15, ""weighted"": fal..."
2,6 ALS + признаки,10,0.0085,0.0125,0.0132,52.2000,"{""factors"": 32, ""regularization"": 0.009759441755448826, ""alpha"": 46.008165366548866, ""iterations"": 15, ""weighted"": f..."
3,6 ALS + признаки,15,0.0085,0.0126,0.0136,52.0000,"{""factors"": 32, ""regularization"": 0.001408902905163939, ""alpha"": 98.54824474988237, ""iterations"": 15, ""weighted"": fa..."
4,6 ALS + признаки,13,0.0084,0.0125,0.0132,53.0000,"{""factors"": 32, ""regularization"": 0.2687882348581276, ""alpha"": 17.764477127240685, ""iterations"": 15, ""weighted"": fal..."


{'knn': {'similarity': 'bm25',
  'K': 565,
  'weighted': True,
  'K1': 0.3355838788284944,
  'B': 0.28920875241277766},
 'als': {'factors': 128,
  'regularization': 0.0013995874675233253,
  'alpha': 82.20322733155712,
  'iterations': 23,
  'weighted': True},
 'als_features': {'factors': 32,
  'regularization': 0.0064398121586677305,
  'alpha': 68.3964886529918,
  'iterations': 15,
  'weighted': False,
  'use': ['author'],
  'min_author_books': 10,
  'together': False}}

**Подбор (80 попыток, 56 мин; время каждой — в `cf_trials.csv`).** Подбор шёл до правила кандидатов (книги с 1–4
оценками в обучении не рекомендуются, 3.1) — значения NDCG@10 в этой таблице без правила; параметры не переподбирались.
- **ItemKNN** — BM25 лучше cosine и TF-IDF во всех попытках (лучшее 0,0111 против 0,0095 и 0,0099). Лучшие — большое
  число соседей (K 400–850) и маленькие K1, B (~0,3, у нижней границы пространства); веса оценок чуть помогают.
  Попытка — 4–6 с, весь подбор — 1,8 мин.
- **ALS** — вдвое лучше ALS по умолчанию (0,0083 против 0,0042). Главное — `alpha` 50–85 вместо 1 (вес наблюдаемых
  оценок) и 128 факторов (TPE отдал им 22 попытки из 30; 256 — хуже, 0,0049); регуляризация почти не влияет
  (0,001–0,025 — одинаково), итераций хватает 20–26. Попытка — ~20 с, подбор — 10 мин.
- **ALS + признаки** — помогают только частые авторы (≥ 10 книг): раздел и подраздел портят (лучшее 0,0055 и 0,0076
  против 0,0086 у одних авторов). Лучше 32 фактора, без весов оценок, признаки отдельно от факторов. Попытка — ~2 мин
  (признаки — плотные столбцы), подбор — 44 мин.
- Часть лучших значений у границ пространства (K1, B, регуляризация), но по соседним попыткам там плато — расширять не стали.

## 3. Лучшие параметры на валидации

In [4]:
for kind, p in BEST.items():
    reco, seconds = fit_recommend(kind, p, train, users_val, books, k=K_GEN)
    record("val", NAMES[kind], reco, seconds, train, val, users_val)

cols = ["модель", "разрез", "пользователей", "NDCG@10", "NDCG@10 95% ДИ", "Recall@10", "Precision@10", "Coverage@10", "Novelty@10"]
fmt = {"пользователей": "{:,}", "NDCG@10": "{:.4f}", "Recall@10": "{:.4f}", "Precision@10": "{:.4f}", "Coverage@10": "{:.4f}",
       "Novelty@10": "{:.2f}"}
res_val = pd.concat(results["val"], ignore_index=True)[cols]
res_val.to_csv(REPORTS / "cf_val.csv", index=False, encoding="utf-8")
display(res_val.style.format(fmt, na_rep="").hide(axis="index"))
display(res_val.pivot_table(index="модель", columns="разрез", values="NDCG@10", sort=False)[["все", "старые", "новые"]])

4 ItemKNN: 6 с, NDCG@10 все / старые / новые: 0.0105 / 0.0160 / 0.0000


5 ALS: 29 с, NDCG@10 все / старые / новые: 0.0083 / 0.0122 / 0.0000


  0%|          | 0/1 [00:00<?, ?it/s]

6 ALS + признаки: 77 с, NDCG@10 все / старые / новые: 0.0085 / 0.0124 / 0.0000


модель,разрез,пользователей,NDCG@10,NDCG@10 95% ДИ,Recall@10,Precision@10,Coverage@10,Novelty@10
1 Popularity,все,"21,350",0.0024,0.0020–0.0028,0.0051,0.0010,0.0002,6.82
1 Popularity,старые,"15,945",0.0035,0.0030–0.0041,0.0076,0.0013,,
1 Popularity,новые,"9,396",0.0000,0.0000–0.0000,0.0000,0.0000,,
2 Content TF-IDF,все,"21,350",0.0107,0.0098–0.0116,0.0188,0.0039,0.7946,15.63
2 Content TF-IDF,старые,"15,945",0.0054,0.0046–0.0062,0.0098,0.0018,,
2 Content TF-IDF,новые,"9,396",0.0176,0.0158–0.0193,0.0324,0.0059,,
3 Content e5-base,все,"21,350",0.0099,0.0090–0.0109,0.0162,0.0034,0.6880,15.57
3 Content e5-base,старые,"15,945",0.0049,0.0041–0.0057,0.0084,0.0016,,
3 Content e5-base,новые,"9,396",0.0167,0.0149–0.0185,0.0285,0.0051,,
ALS по умолчанию,все,"21,350",0.0042,0.0037–0.0047,0.0070,0.0018,0.0106,8.84


разрез,все,старые,новые
модель,,,
1 Popularity,0.0024,0.0035,0.0000
2 Content TF-IDF,0.0107,0.0054,0.0176
3 Content e5-base,0.0099,0.0049,0.0167
ALS по умолчанию,0.0042,0.0061,0.0000
4 ItemKNN,0.0105,0.0160,0.0000
5 ALS,0.0083,0.0122,0.0000
6 ALS + признаки,0.0085,0.0124,0.0000


In [5]:
def significance(part_name, pairs):
    rows = []
    for a, b, s in pairs:
        r = paired_bootstrap(per_user[part_name][a][s], per_user[part_name][b][s])
        rows.append({"сравнение": f"{a} − {b}", "разрез": s, "разница NDCG@10": r["разница"],
                     "95% ДИ": f"{r['low']:+.4f} … {r['high']:+.4f}", "значимо": "да" if r["low"] > 0 or r["high"] < 0 else "нет"})
    return pd.DataFrame(rows).style.format({"разница NDCG@10": "{:+.4f}"}).hide(axis="index")

PAIRS = [(m, ref, s) for m in NAMES.values() for ref in ["2 Content TF-IDF", "ALS по умолчанию"] for s in ["все", "старые"]]
PAIRS += [("6 ALS + признаки", "5 ALS", "все"), ("5 ALS", "4 ItemKNN", "все"), ("5 ALS", "4 ItemKNN", "старые")]
significance("val", PAIRS)

сравнение,разрез,разница NDCG@10,95% ДИ,значимо
4 ItemKNN − 2 Content TF-IDF,все,-0.0001,-0.0012 … +0.0010,нет
4 ItemKNN − 2 Content TF-IDF,старые,+0.0106,+0.0094 … +0.0119,да
4 ItemKNN − ALS по умолчанию,все,+0.0063,+0.0053 … +0.0073,да
4 ItemKNN − ALS по умолчанию,старые,+0.0098,+0.0084 … +0.0113,да
5 ALS − 2 Content TF-IDF,все,-0.0024,-0.0035 … -0.0013,да
5 ALS − 2 Content TF-IDF,старые,+0.0068,+0.0056 … +0.0081,да
5 ALS − ALS по умолчанию,все,+0.0041,+0.0033 … +0.0048,да
5 ALS − ALS по умолчанию,старые,+0.0061,+0.0050 … +0.0072,да
6 ALS + признаки − 2 Content TF-IDF,все,-0.0022,-0.0033 … -0.0011,да
6 ALS + признаки − 2 Content TF-IDF,старые,+0.0070,+0.0057 … +0.0082,да


**Валидация (с правилом кандидатов; content — из всего каталога, см. раздел 5):**
- **Правило стоит ItemKNN немного:** 0,0111 → 0,0105 (на книги с 1–4 оценками в обучении приходится 16% попаданий
  валидации на «старых» — они попали в подвыборку благодаря будущим оценкам). ALS не изменился (0,0083): редкие книги он и так не советует.
- **На «все» ItemKNN = TF-IDF** (0,0105 и 0,0107, незначимо), но на «старых» ItemKNN в 3 раза лучше (0,0160 против
  0,0054), на «новых» — 0. Content на «все» держится на новых книгах — и это почти целиком утечка каталога (раздел 5).
- **ItemKNN значимо лучше ALS** (−0,0023 на «все», −0,0037 на «старых») и в 5 раз быстрее.
- **Признаки книг ALS не помогают:** +0,0002, незначимо; покрытие выросло (0,19 против 0,09), качество — нет.

## 4. Тест — один прогон

Обучение — все оценки до 01.01.2022 (обучение + валидация), проверка — 2022–2023. Параметры — лучшие с валидации, без
изменений. «Старые» книги на тесте — с оценками до 2022 года.

In [6]:
train_full = pd.concat([train, val])
users_test = eval_users(train_full, test)
print(f"Пользователей для проверки на тесте: {len(users_test):,}; книг обучения: {train_full[Columns.Item].nunique():,}")

for name, make in BASE.items():
    t0 = time.time()
    reco = make(train_full, users_test)
    record("test", name, reco, time.time() - t0, train_full, test, users_test)
for kind, p in BEST.items():
    reco, seconds = fit_recommend(kind, p, train_full, users_test, books, k=K_GEN)
    record("test", NAMES[kind], reco, seconds, train_full, test, users_test)

res_test = pd.concat(results["test"], ignore_index=True)[cols]
res_test.to_csv(REPORTS / "cf_test.csv", index=False, encoding="utf-8")
display(res_test.style.format(fmt, na_rep="").hide(axis="index"))
display(res_test.pivot_table(index="модель", columns="разрез", values="NDCG@10", sort=False)[["все", "старые", "новые"]])

Пользователей для проверки на тесте: 20,009; книг обучения: 86,590


1 Popularity: 1 с, NDCG@10 все / старые / новые: 0.0016 / 0.0024 / 0.0000


2 Content TF-IDF: 321 с, NDCG@10 все / старые / новые: 0.0108 / 0.0045 / 0.0195


3 Content e5-base: 36 с, NDCG@10 все / старые / новые: 0.0089 / 0.0045 / 0.0149


ALS по умолчанию: 15 с, NDCG@10 все / старые / новые: 0.0038 / 0.0054 / 0.0000


4 ItemKNN: 8 с, NDCG@10 все / старые / новые: 0.0106 / 0.0160 / 0.0000


5 ALS: 28 с, NDCG@10 все / старые / новые: 0.0083 / 0.0122 / 0.0000


  0%|          | 0/1 [00:00<?, ?it/s]

6 ALS + признаки: 83 с, NDCG@10 все / старые / новые: 0.0074 / 0.0108 / 0.0000


модель,разрез,пользователей,NDCG@10,NDCG@10 95% ДИ,Recall@10,Precision@10,Coverage@10,Novelty@10
1 Popularity,все,"20,009",0.0016,0.0013–0.0019,0.0033,0.0007,0.0002,6.81
1 Popularity,старые,"15,194",0.0024,0.0020–0.0028,0.0049,0.0010,,
1 Popularity,новые,"8,739",0.0000,0.0000–0.0000,0.0000,0.0000,,
2 Content TF-IDF,все,"20,009",0.0108,0.0099–0.0119,0.0168,0.0044,0.7297,15.59
2 Content TF-IDF,старые,"15,194",0.0045,0.0038–0.0053,0.0073,0.0016,,
2 Content TF-IDF,новые,"8,739",0.0195,0.0174–0.0216,0.0320,0.0073,,
3 Content e5-base,все,"20,009",0.0089,0.0080–0.0098,0.0139,0.0037,0.6158,15.47
3 Content e5-base,старые,"15,194",0.0045,0.0038–0.0052,0.0074,0.0016,,
3 Content e5-base,новые,"8,739",0.0149,0.0131–0.0168,0.0246,0.0057,,
ALS по умолчанию,все,"20,009",0.0038,0.0033–0.0043,0.0057,0.0018,0.0105,8.90


разрез,все,старые,новые
модель,,,
1 Popularity,0.0016,0.0024,0.0000
2 Content TF-IDF,0.0108,0.0045,0.0195
3 Content e5-base,0.0089,0.0045,0.0149
ALS по умолчанию,0.0038,0.0054,0.0000
4 ItemKNN,0.0106,0.0160,0.0000
5 ALS,0.0083,0.0122,0.0000
6 ALS + признаки,0.0074,0.0108,0.0000


In [7]:
significance("test", PAIRS)

сравнение,разрез,разница NDCG@10,95% ДИ,значимо
4 ItemKNN − 2 Content TF-IDF,все,-0.0002,-0.0015 … +0.0011,нет
4 ItemKNN − 2 Content TF-IDF,старые,+0.0115,+0.0101 … +0.0128,да
4 ItemKNN − ALS по умолчанию,все,+0.0069,+0.0059 … +0.0079,да
4 ItemKNN − ALS по умолчанию,старые,+0.0105,+0.0092 … +0.0119,да
5 ALS − 2 Content TF-IDF,все,-0.0025,-0.0038 … -0.0013,да
5 ALS − 2 Content TF-IDF,старые,+0.0077,+0.0063 … +0.0089,да
5 ALS − ALS по умолчанию,все,+0.0045,+0.0038 … +0.0053,да
5 ALS − ALS по умолчанию,старые,+0.0067,+0.0056 … +0.0078,да
6 ALS + признаки − 2 Content TF-IDF,все,-0.0034,-0.0046 … -0.0022,да
6 ALS + признаки − 2 Content TF-IDF,старые,+0.0063,+0.0051 … +0.0075,да


In [8]:
pd.DataFrame(timing).pivot_table(index="модель", columns="часть", values="обучение + рекомендации, с", sort=False)

часть,val,test
модель,,
1 Popularity,1.1000,1.4000
2 Content TF-IDF,337.8000,320.9000
3 Content e5-base,37.1000,35.9000
ALS по умолчанию,15.8000,14.8000
4 ItemKNN,6.1000,8.2000
5 ALS,28.6000,28.3000
6 ALS + признаки,77.2000,83.4000


**Тест (один прогон, параметры с валидации, с правилом кандидатов):**
- **Порядок:** TF-IDF 0,0108 (весь каталог — завышено), ItemKNN 0,0106, e5-base 0,0089, ALS 0,0083, ALS + признаки
  0,0074, ALS по умолчанию 0,0038, Popularity 0,0016. ItemKNN = TF-IDF на «все» (незначимо), на «старых» в 3,6 раза
  лучше (0,0160 против 0,0045); ItemKNN значимо лучше ALS (−0,0024); признаки ALS на тесте значимо хуже ALS (−0,0009).
- **Переобучения на валидации нет:** ItemKNN 0,0105 → 0,0106, ALS 0,0083 → 0,0083; ALS + признаки просел
  (0,0085 → 0,0074) — его выигрыш на валидации был шумом.
- **Время** (обучение + рекомендации 20 тыс. пользователей, процессор): ItemKNN 8 с, ALS 28 с, ALS + признаки 83 с,
  TF-IDF ~5,5 мин (профили по 200 тыс. признаков на весь каталог), e5-base 36 с.
- **Для 3.3:** коллаборативная часть гибрида — ItemKNN (BM25), content — TF-IDF и e5-base, ALS — источник кандидатов (E3).

## 5. Content-модели на каталоге «известные на дату»

В разделах 1–4 content-модели выбирают из всего каталога `books_clean` — а он собран из книг, у которых есть отзывы,
в том числе после даты разбиения. Книга, впервые оценённая в 2021 году, на 01.01.2021 «знает», что её оценят: это утечка.
Здесь — тот же расчёт на каталоге **известных на дату** книг: первая оценка в полных данных раньше даты разбиения
(валидация — до 2021, тест — до 2022). ItemKNN и ALS не затронуты — выбирают только из книг обучения.

In [9]:
rows_known, pu_known = [], {}
for part_name, tr, part, us, cutoff in [("val", train, val, users_val, VAL_START),
                                         ("test", train_full, test, users_test, TEST_START)]:
    allowed_known = allowed_all & known_at(ids_all, first, cutoff)
    print(f"{part_name}: известных на {cutoff:%d.%m.%Y} книг — {allowed_known.sum():,} из {allowed_all.sum():,} ({allowed_known.sum() / allowed_all.sum():.1%})")
    for name, vectors in [("2 Content TF-IDF", X_tfidf), ("3 Content e5-base", E5)]:
        t0 = time.time()
        reco = drop_rare(recommend(vectors, ids_all, tr, us, k=K_GEN, allowed=allowed_known), tr, k=K)
        table, pu = evaluate(reco, tr, part, us)
        pu_known[(part_name, name)] = pu
        rows_known.append(with_ci(table, pu).assign(модель=f"{name} (известные)", часть=part_name))
        print(f"  {name}: {time.time() - t0:.0f} с, NDCG@10 все / старые / новые: " + " / ".join(f"{v:.4f}" for v in table["NDCG@10"]))

res_known = pd.concat(rows_known, ignore_index=True)
res_known[["часть"] + cols].to_csv(REPORTS / "cf_content_known.csv", index=False, encoding="utf-8")
for part_name, res in [("val", res_val), ("test", res_test)]:
    both = pd.concat([res, res_known[res_known["часть"] == part_name][cols]])
    print(f"NDCG@10, {part_name}:")
    display(both.pivot_table(index="модель", columns="разрез", values="NDCG@10", sort=False)[["все", "старые", "новые"]])

val: известных на 01.01.2021 книг — 376,233 из 405,555 (92.8%)


  2 Content TF-IDF: 343 с, NDCG@10 все / старые / новые: 0.0042 / 0.0060 / 0.0002


  3 Content e5-base: 38 с, NDCG@10 все / старые / новые: 0.0038 / 0.0055 / 0.0001


test: известных на 01.01.2022 книг — 390,604 из 405,555 (96.3%)


  2 Content TF-IDF: 323 с, NDCG@10 все / старые / новые: 0.0035 / 0.0049 / 0.0004


  3 Content e5-base: 43 с, NDCG@10 все / старые / новые: 0.0033 / 0.0048 / 0.0002
NDCG@10, val:


разрез,все,старые,новые
модель,,,
1 Popularity,0.0024,0.0035,0.0000
2 Content TF-IDF,0.0107,0.0054,0.0176
3 Content e5-base,0.0099,0.0049,0.0167
ALS по умолчанию,0.0042,0.0061,0.0000
4 ItemKNN,0.0105,0.0160,0.0000
5 ALS,0.0083,0.0122,0.0000
6 ALS + признаки,0.0085,0.0124,0.0000
2 Content TF-IDF (известные),0.0042,0.0060,0.0002
3 Content e5-base (известные),0.0038,0.0055,0.0001


NDCG@10, test:


разрез,все,старые,новые
модель,,,
1 Popularity,0.0016,0.0024,0.0000
2 Content TF-IDF,0.0108,0.0045,0.0195
3 Content e5-base,0.0089,0.0045,0.0149
ALS по умолчанию,0.0038,0.0054,0.0000
4 ItemKNN,0.0106,0.0160,0.0000
5 ALS,0.0083,0.0122,0.0000
6 ALS + признаки,0.0074,0.0108,0.0000
2 Content TF-IDF (известные),0.0035,0.0049,0.0004
3 Content e5-base (известные),0.0033,0.0048,0.0002


In [10]:
rows = []
for part_name in ["val", "test"]:
    for name in ["2 Content TF-IDF", "3 Content e5-base"]:
        for s in ["все", "старые"]:
            r = paired_bootstrap(per_user[part_name]["4 ItemKNN"][s], pu_known[(part_name, name)][s])
            rows.append({"часть": part_name, "сравнение": f"4 ItemKNN − {name} (известные)", "разрез": s,
                         "разница NDCG@10": r["разница"], "95% ДИ": f"{r['low']:+.4f} … {r['high']:+.4f}",
                         "значимо": "да" if r["low"] > 0 or r["high"] < 0 else "нет"})
pd.DataFrame(rows).style.format({"разница NDCG@10": "{:+.4f}"}).hide(axis="index")

часть,сравнение,разрез,разница NDCG@10,95% ДИ,значимо
val,4 ItemKNN − 2 Content TF-IDF (известные),все,+0.0063,+0.0055 … +0.0072,да
val,4 ItemKNN − 2 Content TF-IDF (известные),старые,+0.0100,+0.0088 … +0.0113,да
val,4 ItemKNN − 3 Content e5-base (известные),все,+0.0068,+0.0060 … +0.0077,да
val,4 ItemKNN − 3 Content e5-base (известные),старые,+0.0104,+0.0092 … +0.0119,да
test,4 ItemKNN − 2 Content TF-IDF (известные),все,+0.0072,+0.0062 … +0.0081,да
test,4 ItemKNN − 2 Content TF-IDF (известные),старые,+0.0111,+0.0097 … +0.0124,да
test,4 ItemKNN − 3 Content e5-base (известные),все,+0.0073,+0.0064 … +0.0083,да
test,4 ItemKNN − 3 Content e5-base (известные),старые,+0.0112,+0.0098 … +0.0126,да


**Content на каталоге «известные на дату» (честный режим):** книга — кандидат, только если её первая оценка (по всем
данным) раньше даты разбиения; каталог — 92,8% книг на 01.01.2021 и 96,3% на 01.01.2022.
- **NDCG@10 падает в 2,5–3 раза:** val TF-IDF 0,0107 → 0,0042, e5-base 0,0099 → 0,0038; test TF-IDF 0,0108 → 0,0035,
  e5-base 0,0089 → 0,0033. На «новых» — почти 0 (0,0001–0,0004): все «новые» попадания content на всём каталоге были
  книгами, впервые оценёнными после даты, — модель выбирала из каталога, который «знает», что эти книги оценят.
- **ItemKNN значимо лучше обеих content-моделей** на «все» и на «старых» (val +0,0063…+0,0068, test +0,0072…+0,0073).
- **Вывод:** офлайн на этих данных content сам по себе слабее коллаборативной модели; польза от него — в гибриде (E3)
  и на сайте для настоящих новых книг (русский и казахский каталог, книги без оценок), которую на этих данных не измерить.